# pandas 로 만드는 현금흐름표 · 실습 노트북

글 · 안상선 ((주)M-Robo 대표) · 파이썬과 회계데이터 분석

1. [재무제표 생성기](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/generator.html)에 **내 학번**을 넣고 `재무제표_학번.csv` 를 내려받는다. 화면의 **확인 코드**를 적어 둔다.
2. 아래 셀을 **위에서부터 차례로** 실행한다 (Shift+Enter). 첫 셀에서 '파일 선택' 단추가 나오면 받은 CSV 를 고른다.
3. 현금흐름표 · 차트 · 재무비율이 나오고, 결과 파일 다섯 개가 내려받아진다.
4. 10장 첫 셀의 출력 전체를 복사해 생성AI 채팅창에 붙여 넣으면 분석보고서가 나온다. 마지막 셀의 검증표로 숫자를 맞춰 본다.

코드는 들여쓰기 · def · for 없이 썼다. 줄마다의 뜻과 엑셀로 하는 방법은 [책의 카드](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#upload)에 있다.

## 5. 내 재무제표 불러오기 · 학번으로 만든 CSV 를 Colab 에 올려 읽는다

0~4장은 (주)SG화학 숫자를 코드에 직접 적었다. 5장부터는 나만의 회사다. 먼저 생성기(generator.html)에 학번을 넣어 재무제표_학번.csv 를 받는다. 같은 학번은 언제나 같은 숫자가 나오고, 학번마다 회사와 숫자가 다르다. Colab 에서는 google.colab 의 files.upload() 를 실행하면 '파일 선택' 단추가 나온다. 받은 CSV 를 고르면 Colab 폴더에 올라가고, 올린 파일 이름이 사전의 키로 돌아온다. list(up)[0] 로 그 이름을 꺼내 pd.read_csv 로 읽는다. 이 카드부터 12장 전까지는 하나의 노트북에서 차례로 실행한다. 앞 셀에서 만든 변수를 다음 셀이 그대로 쓴다.

책의 카드: [upload](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#upload)

In [ ]:
import numpy as np
import pandas as pd
from google.colab import files
pd.set_option("display.unicode.east_asian_width", True)   # 한글 열 줄 맞춤

up = files.upload()                  # '파일 선택' 단추를 눌러 재무제표_학번.csv 를 고른다
name = list(up)[0]                   # 올린 파일 이름
fs = pd.read_csv(name)               # CSV → 표
company = fs["회사명"].iloc[0]
sid = str(fs["학번"].iloc[0])
print("회사:", company, "| 학번:", sid, "| 줄 수:", len(fs))
print(fs[["표", "구분", "세부", "계정과목", "기초", "기말"]])

## 5. 내 재무제표 불러오기 · 한 파일을 재무상태표와 손익계산서로 나눈다

표 열의 값으로 줄을 걸러 두 표로 나눈다. 재무상태표는 0~4장과 같은 모양(계정과목, 구분, 기초, 기말)으로 만들고 증감 열을 붙인다. 손익계산서는 CSV 에 비용도 양수로 적혀 있어서, 구분이 '수익' 이면 그대로, '비용' 이면 − 를 붙여 1장의 pl 과 같은 모양(이름표가 계정과목인 Series)으로 만든다. 그러면 2~4장에서 쓴 식 pl.sum(), pl["감가상각비"], d["매출채권"] 을 그대로 쓸 수 있다.

책의 카드: [split](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#split)

In [ ]:
bs = fs[fs["표"] == "재무상태표"].copy()
bs["기초"] = bs["기초"].astype(int)            # 손익 줄의 빈칸 때문에 실수로 읽힌 것을 정수로
bs["증감"] = bs["기말"] - bs["기초"]
b = bs.set_index("계정과목")                    # b.loc["현금", "기말"] 처럼 꺼내 쓴다
d = b["증감"]                                  # d["매출채권"] 처럼 꺼내 쓴다
print(bs[["계정과목", "구분", "세부", "기초", "기말", "증감"]])

p = fs[fs["표"] == "손익계산서"]
pl = pd.Series(np.where(p["구분"] == "수익", p["기말"], -p["기말"]), index=p["계정과목"].values, name="금액")
ni = pl.sum()                                  # 당기순이익
print(pl)
print("당기순이익:", ni)

## 5. 내 재무제표 불러오기 · 대차는 맞는데, 이익과 현금이 안 맞는다

먼저 대차를 확인한다. 생성기가 만든 숫자는 기초와 기말 모두 자산 = 부채 + 자본 이 맞는다. 그다음 당기순이익과 현금의 증감을 나란히 놓는다. 두 숫자는 거의 언제나 다르다. 이익이 났는데 현금이 줄기도 한다. 이 차이를 계단으로 설명하는 것이 6장의 현금흐름표다.

책의 카드: [mismatch](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#mismatch)

In [ ]:
side = bs.groupby("구분")[["기초", "기말"]].sum()
gap = side.loc["자산"] - side.loc["부채"] - side.loc["자본"]
print(side)
print("대차가 맞는가:", gap.abs().sum() == 0)

print("당기순이익:", ni)
print("현금의 증감:", d["현금"], "(", b.loc["현금", "기초"], "→", b.loc["현금", "기말"], ")")
print("차이 (현금 증감 - 순이익):", d["현금"] - ni)

## 6. 내 회사의 현금흐름표 · 2~4장의 식을 그대로 써서 내 회사의 현금흐름표를 만든다

계정 이름이 SG화학과 같으므로 2~4장에서 배운 식을 그대로 쓴다. 표를 빈 DataFrame 으로 만들고 활동, 항목, 금액 세 열을 차례로 채운다. ① 당기순이익 ② 감가상각비 가산 ③ 처분이익 차감 ④ 영업 관련 자산·부채 다섯 개(자산 증가 −, 부채 증가 +)가 영업활동이고, 유형자산 취득과 무형자산 처분이 투자활동, 차입금 순증감과 유상증자(자본금 증감)와 배당금이 재무활동이다. 배당금은 이익잉여금에서 거꾸로 구한다.

책의 카드: [my_cf](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#my_cf)

In [ ]:
ops = ["매출채권", "상품재고", "선급보험료", "매입채무", "미지급급여"]
wc = b.loc[ops]
wc_effect = np.where(wc["구분"] == "자산", -wc["증감"], wc["증감"])     # 자산 증가는 −, 부채 증가는 +
gain = pl["무형자산처분이익"]
dividend = b.loc["이익잉여금", "기초"] + ni - b.loc["이익잉여금", "기말"]

cf = pd.DataFrame()
cf["활동"] = ["영업활동"] * 8 + ["투자활동"] * 2 + ["재무활동"] * 3
cf["항목"] = ["① 당기순이익", "② 감가상각비 가산", "③ 무형자산처분이익 차감", "④ 매출채권 증감", "④ 상품재고 증감",
"④ 선급보험료 증감", "④ 매입채무 증감", "④ 미지급급여 증감", "유형자산의 취득", "무형자산의 처분",
"차입금의 순증감", "유상증자", "배당금의 지급"]
cf["금액"] = [ni, -pl["감가상각비"], -gain] + list(wc_effect) + [-d["유형자산"], -d["무형자산"] + gain, d["차입금"], d["자본금"], -dividend]
cf["금액"] = cf["금액"].astype(int)
print(cf)
total = cf.groupby("활동", sort=False)["금액"].sum()
print(total)

## 6. 내 회사의 현금흐름표 · 검산하고, 이익과 현금이 다른 이유 세 가지를 찾는다

먼저 4장의 검산을 한다. 세 활동의 합이 현금 증감과 같아야 한다. 그다음 순이익과 영업현금의 차이를 만든 조정 항목(②~④) 중 크기가 큰 세 개를 고른다. 금액의 절댓값(abs)으로 크기 열을 만들고 큰 순서로 정렬해 위의 세 줄을 본다. 이 세 줄이 5장에서 본 '이익과 현금이 안 맞는' 이유이고, 10장 보고서의 핵심 문장이 된다.

책의 카드: [why_diff](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#why_diff)

In [ ]:
print("세 활동 합계:", total.sum(), "| 현금 증감:", d["현금"], "| 검산 통과:", total.sum() == d["현금"])
cfo = total["영업활동"]
print("당기순이익:", ni, "→ 영업활동 현금흐름:", cfo, "| 차이:", cfo - ni)

adj = cf[cf["항목"].str[0].isin(["②", "③", "④"])].copy()    # 조정 항목만
adj["크기"] = adj["금액"].abs()
top3 = adj.sort_values("크기", ascending=False).head(3)
print(top3[["항목", "금액"]])

## 7. 차트로 보기 · 재무상태표의 구조를 쌓은 막대로

기초와 기말에 자산 막대와 부채+자본 막대를 하나씩 세운다. 자산 막대는 유동·비유동, 부채+자본 막대는 유동부채·비유동부채·자본으로 쌓는다. 두 막대의 높이가 같으면 대차가 맞는 것이다. 막대를 쌓을 때는 bottom 에 아래 칸의 높이를 준다. 차트 글자는 Colab 에서 깨지지 않게 영어로 적는다.

책의 카드: [bs_chart](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#bs_chart)

In [ ]:
import matplotlib.pyplot as plt

sumif = bs.groupby(["구분", "세부"])[["기초", "기말"]].sum()     # 구분·세부별 합계
print(sumif)
ca = sumif.loc[("자산", "유동")]
nca = sumif.loc[("자산", "비유동")]
cl = sumif.loc[("부채", "유동")]
ncl = sumif.loc[("부채", "비유동")]
eq = sumif.loc[("자본", "자본")]

x = ["Assets\n(open)", "L+E\n(open)", "Assets\n(close)", "L+E\n(close)"]
lower = [ca["기초"], cl["기초"], ca["기말"], cl["기말"]]           # 아래 칸: 유동자산 / 유동부채
middle = [nca["기초"], ncl["기초"], nca["기말"], ncl["기말"]]      # 가운데 칸: 비유동자산 / 비유동부채
upper = [0, eq["기초"], 0, eq["기말"]]                              # 위 칸: 자본

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.bar(x, lower, color="#4472C4", label="Current")
ax.bar(x, middle, bottom=lower, color="#A9BEE3", label="Non-current")
ax.bar(x, upper, bottom=np.add(lower, middle), color="#2E9E5B", label="Equity")
ax.legend(loc="upper left", fontsize=8)
ax.set(title=f"Balance sheet structure (student {sid}, KRW mn)")
plt.show()

## 7. 차트로 보기 · 순이익에서 영업현금까지 폭포 그리기

간접법 계단을 그림으로 본다. 첫 막대는 당기순이익, 그다음 일곱 막대는 조정 항목, 마지막 막대는 영업활동 현금흐름이다. 조정 막대는 직전까지의 누계에서 시작한다. np.cumsum 은 누계를 한 번에 계산하고, 그 누계를 한 칸 밀면 각 막대의 시작 높이(bottom)가 된다. 늘린 막대는 초록, 줄인 막대는 빨강이다.

책의 카드: [ni_to_cfo](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#ni_to_cfo)

In [ ]:
op = cf[cf["활동"] == "영업활동"]
steps = op["금액"].values                      # 순이익 + 조정 일곱 개
run = np.cumsum(steps)                         # 누계
bottom = np.concatenate([[0, steps[0]], run[1:-1], [0]])     # 막대가 시작하는 높이
height = np.concatenate([[steps[0]], steps[1:], [run[-1]]])   # 막대의 길이 (마지막은 영업현금 전체)
labels = ["Net income", "Depreciation", "Disposal gain", "Receivables", "Inventory", "Prepaid", "Payables", "Wages payable", "Operating cash"]
colors = ["#1F3A5F"] + list(np.where(steps[1:] >= 0, "#2E9E5B", "#C9382A")) + ["#1F3A5F"]
print(pd.DataFrame({"bar": labels, "bottom": bottom, "height": height}))

fig, ax = plt.subplots(figsize=(7.5, 3.8))
bars = ax.bar(labels, height, bottom=bottom, color=colors)
ax.bar_label(bars, labels=list(np.char.mod("%+d", height)), fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)
ax.tick_params(axis="x", labelsize=7.5, labelrotation=30)
ax.set(title=f"Net income to operating cash flow (student {sid})")
plt.show()

## 7. 차트로 보기 · 기초 현금에서 기말 현금까지 다리 놓기

4장 waterfall 카드와 같은 그림을 내 회사 숫자로 그린다. 기초 현금에서 영업·투자·재무 세 활동을 차례로 더하면 기말 현금이다. 세 활동의 부호 조합(+ − + 처럼)이 회사가 지금 어떤 상황인지 보여 준다. 10장 보고서에서 이 조합을 해석한다.

책의 카드: [cash_bridge](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#cash_bridge)

In [ ]:
cash0 = b.loc["현금", "기초"]
cash1 = b.loc["현금", "기말"]
flow = total.values                                   # 영업, 투자, 재무
b_labels = ["Opening", "Operating", "Investing", "Financing", "Closing"]
b_height = np.concatenate([[cash0], flow, [cash1]])
b_bottom = np.concatenate([[0], cash0 + np.cumsum(flow) - flow, [0]])
b_colors = ["#1F3A5F"] + list(np.where(flow >= 0, "#2E9E5B", "#C9382A")) + ["#1F3A5F"]
print(pd.DataFrame({"bar": b_labels, "bottom": b_bottom, "height": b_height}))
print("부호 조합 (영업, 투자, 재무):", list(np.where(flow >= 0, "+", "-")))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
bars = ax.bar(b_labels, b_height, bottom=b_bottom, color=b_colors)
ax.bar_label(bars, labels=[str(cash0)] + list(np.char.mod("%+d", flow)) + [str(cash1)])
ax.axhline(0, color="black", linewidth=0.8)
ax.margins(y=0.15)
ax.set(title=f"Cash bridge (student {sid}, KRW mn)")
plt.show()

## 8. 주요 재무비율 · 비율에 들어갈 재료를 먼저 모은다

재무비율은 '재료 ÷ 재료' 다. 재료를 먼저 이름 붙여 모아 두면 비율 식이 말과 같은 모양이 된다. 재무상태표에서는 유동자산·유동부채·총자산·부채·자본의 합계를, 손익계산서에서는 매출총이익·영업이익·이자비용을 만든다. 회전율과 ROA·ROE 는 1년 동안의 흐름(손익)을 한 시점의 잔액으로 나누므로, 기초와 기말의 평균 잔액을 쓴다.

책의 카드: [ratio_parts](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#ratio_parts)

In [ ]:
cur_assets = bs.loc[(bs["구분"] == "자산") & (bs["세부"] == "유동"), "기말"].sum()
cur_liab = bs.loc[(bs["구분"] == "부채") & (bs["세부"] == "유동"), "기말"].sum()
assets0 = bs.loc[bs["구분"] == "자산", "기초"].sum()
assets1 = bs.loc[bs["구분"] == "자산", "기말"].sum()
liab1 = bs.loc[bs["구분"] == "부채", "기말"].sum()
equity0 = bs.loc[bs["구분"] == "자본", "기초"].sum()
equity1 = bs.loc[bs["구분"] == "자본", "기말"].sum()

sales = pl["매출액"]
cogs = -pl["매출원가"]                                    # 비용은 음수로 들어 있어 − 를 붙여 양수로
gross = sales - cogs                                      # 매출총이익
op_income = gross + pl["보험료"] + pl["급여"] + pl["감가상각비"] + pl["기타영업비용"]   # 영업이익
interest = -pl["이자비용"]
avg_assets = (assets0 + assets1) / 2
avg_equity = (equity0 + equity1) / 2
avg_ar = (b.loc["매출채권", "기초"] + b.loc["매출채권", "기말"]) / 2
avg_inv = (b.loc["상품재고", "기초"] + b.loc["상품재고", "기말"]) / 2

parts = pd.Series([cur_assets, cur_liab, assets1, liab1, equity1, sales, gross, op_income, interest, ni, avg_assets, avg_equity, avg_ar, avg_inv, cfo],
index=["유동자산", "유동부채", "총자산(기말)", "부채(기말)", "자본(기말)", "매출액", "매출총이익", "영업이익", "이자비용", "당기순이익",
"평균총자산", "평균자본", "평균매출채권", "평균재고", "영업활동현금흐름"])
print(parts)

## 8. 주요 재무비율 · 주요 재무비율 열세 개를 한 표로

안정성 · 수익성 · 활동성 · 현금흐름 네 묶음, 열세 개의 비율을 한 표에 담는다. 비율마다 계산식을 글로 적어 두면 보고서를 쓸 때 그대로 옮길 수 있다. 비율은 %, 회전율은 회, 배율은 배로 읽는다. round(1) 로 소수 첫째 자리까지 남긴다.

책의 카드: [ratio_table](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#ratio_table)

In [ ]:
ratio = pd.DataFrame()
ratio["분류"] = ["안정성"] * 4 + ["수익성"] * 5 + ["활동성"] * 3 + ["현금흐름"]
ratio["비율"] = ["유동비율", "부채비율", "자기자본비율", "이자보상배율", "매출총이익률", "영업이익률", "순이익률", "ROA", "ROE",
"매출채권회전율", "재고자산회전율", "총자산회전율", "영업현금흐름/순이익"]
ratio["값"] = [cur_assets / cur_liab * 100, liab1 / equity1 * 100, equity1 / assets1 * 100, op_income / interest,
gross / sales * 100, op_income / sales * 100, ni / sales * 100, ni / avg_assets * 100, ni / avg_equity * 100,
sales / avg_ar, cogs / avg_inv, sales / avg_assets, cfo / ni]
ratio["값"] = ratio["값"].round(1)
ratio["단위"] = ["%", "%", "%", "배", "%", "%", "%", "%", "%", "회", "회", "회", "배"]
ratio["계산식"] = ["유동자산 ÷ 유동부채", "부채 ÷ 자본", "자본 ÷ 총자산", "영업이익 ÷ 이자비용", "매출총이익 ÷ 매출액", "영업이익 ÷ 매출액",
"당기순이익 ÷ 매출액", "당기순이익 ÷ 평균총자산", "당기순이익 ÷ 평균자본", "매출액 ÷ 평균매출채권", "매출원가 ÷ 평균재고",
"매출액 ÷ 평균총자산", "영업활동현금흐름 ÷ 당기순이익"]
print(ratio)

## 8. 주요 재무비율 · 기준과 비교해 양호·주의를 붙이고 막대로 본다

비율은 숫자만 보면 좋은지 나쁜지 알기 어렵다. 비율마다 참고 기준을 하나 두고, 높을수록 좋은 비율은 '이상', 낮을수록 좋은 비율(부채비율)은 '이하' 로 판정한다. np.where 를 두 번 겹쳐 쓰면 엑셀의 IF 안에 IF 를 넣은 것과 같다. 기준은 교육용 어림값이다. 업종마다 다르므로 보고서에서는 '참고 기준' 이라고 밝힌다. % 단위인 비율만 골라 기준과 나란히 막대로 그린다.

책의 카드: [ratio_signal](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#ratio_signal)

In [ ]:
ratio["기준"] = [150, 200, 40, 3, 25, 5, 3, 3, 8, 6, 6, 1, 1]
ratio["방향"] = ["이상", "이하", "이상", "이상", "이상", "이상", "이상", "이상", "이상", "이상", "이상", "이상", "이상"]
ratio["판정"] = np.where(ratio["방향"] == "이상", np.where(ratio["값"] >= ratio["기준"], "양호", "주의"), np.where(ratio["값"] <= ratio["기준"], "양호", "주의"))
print(ratio[["분류", "비율", "값", "단위", "기준", "방향", "판정"]])
print("주의:", ratio.loc[ratio["판정"] == "주의", "비율"].tolist())

pct = ratio[ratio["단위"] == "%"].copy()
pct["eng"] = ["Current", "Debt/Equity", "Equity/Assets", "Gross margin", "Op. margin", "Net margin", "ROA", "ROE"]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(pct["eng"], pct["값"], color=np.where(pct["판정"] == "양호", "#2E9E5B", "#C9382A"))
ax.scatter(pct["기준"], pct["eng"], color="black", marker="|", s=300, label="Reference")
ax.invert_yaxis()
ax.legend(loc="lower right", fontsize=8)
ax.set(title=f"Ratios in % vs reference (student {sid})", xlabel="%")
plt.show()

## 9. 저장하고 내려받기 · 현금흐름표와 재무비율을 CSV 로 저장해 내려받는다

to_csv 는 표를 CSV 파일로 저장한다. encoding="utf-8-sig" 를 주면 엑셀에서 열어도 한글이 깨지지 않는다. index=False 는 왼쪽의 줄 번호를 빼고 저장한다. 파일 이름에 학번을 넣어 두면 과제 파일이 섞이지 않는다. files.download 는 Colab 폴더의 파일을 내 컴퓨터로 내려받게 한다. 저장한 파일을 다시 읽어 숫자가 그대로인지 확인한다.

책의 카드: [save_csv](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#save_csv)

In [ ]:
cf_name = f"현금흐름표_{sid}.csv"
ratio_name = f"재무비율_{sid}.csv"
cf.to_csv(cf_name, index=False, encoding="utf-8-sig")
ratio.to_csv(ratio_name, index=False, encoding="utf-8-sig")
files.download(cf_name)
files.download(ratio_name)

check_cf = pd.read_csv(cf_name)                     # 다시 읽어 확인
print(cf_name, "→", len(check_cf), "줄, 합계", check_cf["금액"].sum(), "(현금 증감", d["현금"], ")")
print(ratio_name, "→", len(pd.read_csv(ratio_name)), "줄")

## 9. 저장하고 내려받기 · 차트 네 장을 한 그림으로 모아 저장한다

plt.subplots(2, 2) 는 그림 한 장에 칸 네 개를 만든다. axes[0, 0] 은 왼쪽 위 칸이다. 7·8장의 차트를 한 칸씩 다시 그려 한 장으로 모으고 savefig 로 PNG 파일을 만든다. 보고서에 붙이거나 생성AI 에 함께 올릴 수 있다. 칸마다 그리는 줄은 7·8장과 같고, ax 자리만 axes[행, 열] 로 바뀌었다.

책의 카드: [save_png](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#save_png)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes[0, 0].bar(x, lower, color="#4472C4")
axes[0, 0].bar(x, middle, bottom=lower, color="#A9BEE3")
axes[0, 0].bar(x, upper, bottom=np.add(lower, middle), color="#2E9E5B")
axes[0, 0].set(title="Balance sheet structure")
axes[0, 1].bar(b_labels, b_height, bottom=b_bottom, color=b_colors)
axes[0, 1].axhline(0, color="black", linewidth=0.8)
axes[0, 1].set(title="Cash bridge")
op_labels = ["NI", "Dep", "Gain", "AR", "Inv", "Pre", "AP", "Wage", "CFO"]
op_height = np.concatenate([[steps[0]], steps[1:], [run[-1]]])
op_bottom = np.concatenate([[0, steps[0]], run[1:-1], [0]])
op_colors = ["#1F3A5F"] + list(np.where(steps[1:] >= 0, "#2E9E5B", "#C9382A")) + ["#1F3A5F"]
axes[1, 0].bar(op_labels, op_height, bottom=op_bottom, color=op_colors)
axes[1, 0].axhline(0, color="black", linewidth=0.8)
axes[1, 0].set(title="Net income to operating cash")
axes[1, 1].barh(pct["eng"], pct["값"], color=np.where(pct["판정"] == "양호", "#2E9E5B", "#C9382A"))
axes[1, 1].scatter(pct["기준"], pct["eng"], color="black", marker="|", s=200)
axes[1, 1].invert_yaxis()
axes[1, 1].set(title="Ratios in % vs reference")
fig.suptitle(f"Student {sid} (KRW mn)")
fig.tight_layout()
png_name = f"차트_{sid}.png"
fig.savefig(png_name, dpi=150)
files.download(png_name)
plt.show()
print("저장:", png_name)

## 10. 생성AI 로 분석보고서 · 정해진 프롬프트에 내 숫자를 붙여 한 덩어리로 출력한다

생성AI 에게는 계산을 시키지 않는다. 계산은 이미 pandas 가 끝냈다. 생성AI 는 그 숫자를 읽고 글로 풀어 쓰는 일만 한다. 정해진 프롬프트(REPORT) 뒤에 회사 정보, 원래 재무제표, 현금흐름표, 재무비율을 to_csv 로 글자로 바꿔 붙인다. 출력된 글 전체를 복사해 생성AI(ChatGPT, Claude, Gemini 등 아무 채팅창)에 붙여 넣으면 분석보고서가 나온다. 같은 글을 txt 파일로도 저장해 내려받는다. 아래 출력은 표본 학번의 앞부분과 끝부분이다.

책의 카드: [ai_prompt](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#ai_prompt)

In [ ]:
REPORT = """너는 기업 재무분석가다. 아래 [자료]는 학번으로 만든 가상 회사의 재무제표와, 그것으로 파이썬(pandas)이 계산한 간접법 현금흐름표와 재무비율이다.
이 자료만 써서 한국어 분석보고서를 써라.

[지킬 것]
1. 자료에 있는 숫자만 쓴다. 새 숫자를 지어내지 않는다. 계산이 필요하면 식과 함께 보인다.
2. 단위는 백만원. 비율은 자료의 단위(%, 회, 배)를 그대로 쓴다.
3. 판정의 '기준' 은 교육용 참고값이라고 밝힌다. 업종 평균이라고 쓰지 않는다.
4. 투자 권유나 주가 전망은 쓰지 않는다.
5. 표가 필요하면 마크다운 표로 쓴다. 전체 분량은 A4 두 쪽 안팎.

[보고서 구성]
1. 한 줄 요약
2. 회사 개요: 매출액, 총자산, 당기순이익, 영업활동 현금흐름
3. 이익과 현금은 왜 다른가: 당기순이익과 영업활동 현금흐름의 차이를, 간접법 조정 항목 중 금액이 큰 세 개로 설명
4. 현금흐름의 모양: 영업·투자·재무 세 활동의 부호 조합으로 본 회사의 상황, 차입·유상증자·배당의 역할
5. 수익성: 매출총이익률, 영업이익률, 순이익률, ROA, ROE
6. 안정성: 유동비율, 부채비율, 자기자본비율, 이자보상배율
7. 활동성: 매출채권회전율, 재고자산회전율, 총자산회전율
8. 위험 신호와, 경영진에게 묻고 싶은 질문 세 가지
9. 종합 의견과 이 분석의 한계 (1년치 자료, 가상 회사, 주석 없음, 참고 기준의 한계)

[자료]
"""

info = f"회사명: {company} (학번 {sid} 로 만든 가상 회사), 단위: 백만원, 회계기간: 1년\n"
statements = fs[["표", "구분", "세부", "계정과목", "기초", "기말"]].to_csv(index=False)
cf_text = cf.to_csv(index=False)
ratio_text = ratio[["분류", "비율", "값", "단위", "계산식", "기준", "방향", "판정"]].to_csv(index=False)
prompt = REPORT + "[회사]\n" + info + "\n[재무제표]\n" + statements + "\n[현금흐름표 (간접법)]\n" + cf_text + "\n[재무비율]\n" + ratio_text
print(prompt)                                          # ← 이 출력 전체를 복사해 생성AI 에 붙여 넣는다

prompt_name = f"보고서프롬프트_{sid}.txt"
f = open(prompt_name, "w", encoding="utf-8")
f.write(prompt)
f.close()
files.download(prompt_name)

## 10. 생성AI 로 분석보고서 · 생성AI 의 보고서를 정답표로 검산한다

생성AI 는 숫자를 잘못 옮기거나, 시키지 않은 숫자를 지어내기도 한다. 그래서 보고서를 그대로 내지 않는다. 보고서에 꼭 나와야 하는 숫자 열 개를 정답표로 만들어 CSV 로 저장한다. 엑셀에서 열어 '보고서의 숫자' 열을 채우고, 다르면 보고서를 고친다. 이 검증표가 과제 제출물의 하나다.

책의 카드: [ai_verify](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#ai_verify)

In [ ]:
key = pd.DataFrame()
key["항목"] = ["당기순이익", "영업활동 현금흐름", "투자활동 현금흐름", "재무활동 현금흐름", "현금의 증감",
"이익과 현금의 차이 1위 항목", "부채비율(%)", "영업이익률(%)", "ROE(%)", "주의 판정 비율"]
key["정답"] = [ni, total["영업활동"], total["투자활동"], total["재무활동"], d["현금"],
top3["항목"].iloc[0] + " " + str(top3["금액"].iloc[0]), ratio.loc[1, "값"], ratio.loc[5, "값"], ratio.loc[8, "값"],
", ".join(ratio.loc[ratio["판정"] == "주의", "비율"])]
key["보고서의 숫자"] = ""
key["같은가 (O/X)"] = ""
print(key)

key_name = f"보고서검증_{sid}.csv"
key.to_csv(key_name, index=False, encoding="utf-8-sig")
files.download(key_name)

## 다음 할 일

- 10장 첫 셀의 출력(또는 내려받은 `보고서프롬프트_학번.txt`)을 **고치지 않고** 생성AI 에 붙여 넣는다.
- 받은 보고서를 `보고서검증_학번.csv` 로 검산하고 고친다.
- 제출물과 배점은 [과제 안내](https://adminmrobo.github.io/python-accounting-analytics/pandas-cashflow-lab/index.html#homework)를 따른다.